# MWE — Counting a circuit: attributes, gate counts, and resources

"How big is this circuit?" has three different answers in qarp, at three levels of commitment:

1. **Attributes** — the register's shape (`n_qubits`, `n_cbits`, `symbols`). Free, no build needed.
2. **Counters on the built block** — `n_gates()`, `n_2q_gates()`, `depth()`. These count the
   circuit **you wrote**, over `flatten()`.
3. **Stage-explicit resource vectors** (`qarp.resources`) — what will actually *execute*, after
   rebasing to a gate set and routing onto a device's connectivity.

The three disagree, and they are supposed to. This notebook walks the ladder and pins down
the two places people trip: **which commands count as a "gate"**, and **when the number stops
being about your circuit and starts being about the hardware**.

Compilation itself — optimization levels, routers, the DAG passes — is
`mwe_circuit_compilation.ipynb`; the full resource contract is `mwe_resource_estimation.ipynb`.
This notebook is only about *reading numbers off a circuit*.

In [ ]:
from collections import Counter

import sympy

import qarpx as qx
from qarp.blocks import (
    CompositeBlock,
    ConditionalBlock,
    ControlledBlock,
    MeasureBlock,
    ResetBlock,
    SimpleBlock,
    TrotterBlock,
)
from qarp.algorithms import Sampler
from qarp.devices import Device
from qarp.devices import get_nearest_neighbour_architecture
from qarp.operators import QubitOperator
from qarp.resources import Stage, estimate


def workload(n=4):
    # Deliberately mixed: 1q, 2q and 3q gates, plus non-gate commands.
    b = SimpleBlock(n, name="workload")
    b.h(0)
    for k in range(1, n):
        b.cx(0, k)
    b.ccx(0, 1, 2)
    b.rz(3, 0.4)
    b.gphase(0.2)                        # global phase — not a physical gate
    b.reset(1)                           # not a gate either
    b.measure([(q, q) for q in range(n)])
    return b.build()


block = workload()
block

## 1. The register: attributes, not calls

These are plain attributes — no parentheses. Calling `block.n_qubits()` is the most common typo
here; it raises `TypeError: 'int' object is not callable`.

In [ ]:
print("n_qubits:     ", block.n_qubits)        # width of the quantum register
print("n_cbits:      ", block.n_cbits)         # classical register width (from Measure targets)
print("name:         ", block.name)
print("is_built:     ", block.is_built)        # attribute too, not is_built()
print("target_qubits:", block.target_qubits)   # placement in the parent frame
print("symbols:      ", block.symbols)         # canonically sorted tuple, empty here

### ...but "attribute" does not mean "always populated"

The two widths resolve at different times, and only one of them is safe to read before `build()`.

- **`n_qubits` is set in the constructor**, for every block kind: `SimpleBlock` (you passed it),
  `CompositeBlock` (inferred from the children's `target_qubits`), `ControlledBlock`
  (`inner.n_qubits + num_controls`), `ConditionalBlock` (max over its bodies), and
  `MeasureBlock` / `ResetBlock` (both 1). `build()` recomputes it, but never changes it.
- **`n_cbits` is always deferred.** It is derived from the command stream by
  `cbit_register_width(commands_)` at build, so it reads `0` on *any* unbuilt block no matter how
  many measurements you wrote. It also never shrinks: rebuilding does not re-arm an established
  width (conventions §8) — build a fresh block instead.

In [ ]:
def widths(label, blk):
    pre_q, pre_c = blk.n_qubits, blk.n_cbits
    blk.build()
    print(f"{label:<18} n_qubits {pre_q:>2} -> {blk.n_qubits:<2}   n_cbits {pre_c:>2} -> {blk.n_cbits}")


def inner():
    u = SimpleBlock(2, name="U")
    u.h(0)
    u.cx(0, 1)
    return u.build()


measured = SimpleBlock(3, name="measured")
measured.h(0)
measured.measure([(0, 0), (1, 1)])

widths("SimpleBlock", measured)                              # n_cbits only after build
widths("CompositeBlock", CompositeBlock([inner()], name="C"))
widths("ControlledBlock", ControlledBlock(inner(), 2))       # inner + 2 controls, known early
widths("ConditionalBlock", ConditionalBlock([0], [True], inner()))
widths("MeasureBlock", MeasureBlock(0, 0))
widths("ResetBlock", ResetBlock(0))

Primitives are a separate story again: `Sampler`, `StateVector` and the rest are **not** blocks.
They have no width at all until `build()` copies it off the ket — an `AttributeError`, which at
least fails loudly.

In [ ]:
sampler = Sampler(ket=measured, n_shots=100)
try:
    sampler.n_qubits
except AttributeError as exc:
    print("before build ->", exc)
sampler.build()
print("after build  -> n_qubits =", sampler.n_qubits)

`symbols` is the one that carries a contract: it is a tuple **sorted by `str`**, on every built
block, and it is the only correct way to line parameter vectors up with their symbols
(conventions §17). Never hand-zip it against another list — use `block.parameter_map(values)`.

In [ ]:
sym = SimpleBlock(2, name="symbolic")
sym.rz(0, sympy.Symbol("theta"))
sym.cx(0, 1)
sym.ry(1, sympy.Symbol("alpha"))
sym.build()

print("symbols:      ", sym.symbols)           # ('alpha', 'theta') — sorted, not insertion order
print("parameter_map:", sym.parameter_map([0.1, 0.2]))
print("n_gates:      ", sym.n_gates())         # symbolic angles are still gates

## 2. Counting gates

All the counters below share three properties:

- they require `.build()` first (`RuntimeError` otherwise — the command stream isn't defined yet);
- they operate on `flatten()`, so a composite's children are **already included**;
- they count the circuit *as written*, before any transpilation (§6 below).

In [ ]:
print("n_gates():       ", block.n_gates())        # total physical gates, all arities
print("n_1q_gates():    ", block.n_1q_gates())     # shorthand for n_nqb_gates(1)
print("n_2q_gates():    ", block.n_2q_gates())     # shorthand for n_nqb_gates(2)
print("n_nqb_gates(3):  ", block.n_nqb_gates(3))   # the general form: the CCX
print("depth():         ", block.depth())
print()
print("n_gates_of_type(CX):     ", block.n_gates_of_type(qx.GateType.CX))
print("n_gates_of_type(Measure):", block.n_gates_of_type(qx.GateType.Measure))
print()
print("len(flatten()):  ", len(block.flatten()))   # raw command count — nothing excluded

In [ ]:
fresh = SimpleBlock(2)
fresh.h(0)
try:
    fresh.n_gates()
except RuntimeError as exc:
    print("unbuilt ->", exc)

## 3. The gotcha: `n_gates_of_type` is unfiltered, the others are not

`n_gates()` and `n_nqb_gates(k)` count **physical gates** — commands that apply a unitary to the
register. They drop `Barrier`, `Measure`, `Reset`, `GPhase` and the `Branch*` classical-control
markers, via `qx.gate_is_physical`.

`n_gates_of_type(gate)` does no such filtering. It answers "how many commands carry this
`GateType`", counting `Measure` exactly like `CX` — deliberately, so it mirrors
`CircuitDAG.count_ops()`.

So summing `n_gates_of_type` over every type does **not** reproduce `n_gates()`. It reproduces
`len(flatten())`.

In [ ]:
per_type = {g: block.n_gates_of_type(g) for g in qx.GateType.__members__.values()}
total_of_type = sum(per_type.values())
non_physical = sum(1 for c in block.flatten() if not qx.gate_is_physical(c.gate))

print(f"sum(n_gates_of_type over all types) = {total_of_type}")
print(f"len(flatten())                      = {len(block.flatten())}")
print(f"n_gates()                           = {block.n_gates()}")
print(f"non-physical commands               = {non_physical}   (4 Measure + 1 Reset + 1 GPhase)")
print()
print(f"{block.n_gates()} + {non_physical} = {block.n_gates() + non_physical}  ->  the books balance")

In [ ]:
for g in (qx.GateType.H, qx.GateType.CX, qx.GateType.Barrier,
          qx.GateType.Measure, qx.GateType.Reset, qx.GateType.GPhase):
    print(f"{qx.gate_name(g):<8} physical={str(qx.gate_is_physical(g)):<5} "
          f"arity={qx.gate_num_qubits(g)}  params={qx.gate_num_params(g)}")

`gate_num_qubits` is what `n_nqb_gates(k)` buckets on, which gives the other identity worth
remembering: summing the arity buckets *does* recover `n_gates()`, because every physical gate
has exactly one arity.

In [ ]:
buckets = {k: block.n_nqb_gates(k) for k in range(1, block.n_qubits + 1)}
print("arity buckets:", buckets)
print(f"sum = {sum(buckets.values())} == n_gates() = {block.n_gates()}")

## 4. `depth()` is not a count

Depth is the **longest dependency path** through the wire-dependency DAG, not a number of gates.
Gates acting on disjoint qubits share a time step, so four `H`s on four different qubits are four
gates at depth 1, while four `H`s on one qubit are four gates at depth 4.

The weighting differs from the gate counters too: `Barrier` and `GPhase` weigh **0** in depth
(they are fences and bookkeeping, not operations that take time), but `Measure` and `Reset`
**do** occupy a step — unlike in `n_gates()`, where all four are excluded.

In [ ]:
wide = SimpleBlock(4, name="wide")
for q in range(4):
    wide.h(q)
wide.build()

deep = SimpleBlock(4, name="deep")
for _ in range(4):
    deep.h(0)
deep.build()

print(f"wide: n_gates={wide.n_gates()}  depth={wide.depth()}")
print(f"deep: n_gates={deep.n_gates()}  depth={deep.depth()}")

The three exclusions do not line up between the two, which is the second place to be careful.
`GPhase` is invisible to both `n_gates()` and `depth()`; `Measure` and `Reset` are invisible to
`n_gates()` but each occupy a time step:

In [ ]:
def probe(*ops):
    b = SimpleBlock(2, name="probe")
    b.h(0)
    for op in ops:
        op(b)
    b.build()
    return f"depth={b.depth()}  n_gates={b.n_gates()}  commands={len(b.flatten())}"


print("H            ", probe())
print("H + GPhase   ", probe(lambda b: b.gphase(0.2)))    # depth unchanged, n_gates unchanged
print("H + Measure  ", probe(lambda b: b.measure(0, 0)))  # depth +1,       n_gates unchanged
print("H + Reset    ", probe(lambda b: b.reset(0)))       # depth +1,       n_gates unchanged

In [ ]:
dag = qx.CircuitDAG.from_commands(block.flatten())

print("dag.n_nodes:      ", dag.n_nodes, " (every command, physical or not)")
print("dag.depth():      ", dag.depth())
print("block.depth():    ", block.depth(), " (same thing — block.depth() builds this DAG)")
print("len(dag.layers()):", len(dag.layers()), " (ASAP moments == depth)")
print("front_layer():    ", dag.front_layer(), " (node ids ready at t=0)")

## 5. Composites count their children

`flatten()` recurses into the block tree, so a `CompositeBlock`'s counters already include
everything its children contribute — you never sum children by hand.

In [ ]:
child_a = SimpleBlock(2, name="A")
child_a.h(0)
child_a.cx(0, 1)
child_a.build()
child_a.target_qubits = [0, 1]

child_b = SimpleBlock(2, name="B")
child_b.rz(0, 0.3)
child_b.cx(1, 0)
child_b.build()
child_b.target_qubits = [1, 2]        # overlaps A on qubit 1

comp = CompositeBlock([child_a, child_b], name="comp").build()

print(f"child A: n_gates={child_a.n_gates()}   child B: n_gates={child_b.n_gates()}")
print(f"comp:    n_qubits={comp.n_qubits}  n_gates={comp.n_gates()}  "
      f"len(flatten())={len(comp.flatten())}")

## 6. The counts are **pre**-transpilation

This is the substantive caveat. `n_gates()` counts the gates you typed, in whatever gate set you
happened to type them. It is not what runs. `Block.optimize(target_gateset=None, level=1)` lowers
to a gate set (default `qx.native_gateset()`) and simplifies on the DAG, returning a **new**
block — the receiver is untouched.

Note the signature: `level` is the *second* parameter. `optimize(2)` passes `2` as the gate set
and raises; you want `optimize(level=2)`.

In [ ]:
ham = (QubitOperator("X0 X1", 0.5) + QubitOperator("Y0 Y1", 0.5)
       + QubitOperator("Z0 Z1", 0.6))
trotter = TrotterBlock(operator=ham, n_qubits=2, steps=3, time=0.8, order=2).build()

row = f"{'':<12} {'gates':>6} {'1q':>4} {'2q':>4} {'depth':>6}"
print(row + "\n" + "-" * len(row))
print(f"{'as written':<12} {trotter.n_gates():>6} {trotter.n_1q_gates():>4} "
      f"{trotter.n_2q_gates():>4} {trotter.depth():>6}")
for level in (0, 1, 2):
    opt = trotter.optimize(level=level)
    print(f"{'level ' + str(level):<12} {opt.n_gates():>6} {opt.n_1q_gates():>4} "
          f"{opt.n_2q_gates():>4} {opt.depth():>6}")

Level 0 is a pure rebase, so it matches the logical count here; levels 1–2 cancel and merge.
Every level preserves the unitary exactly, global phase included — so these are all honest
counts of the *same* circuit. If you are quoting a two-qubit gate count as a hardware cost,
quote `optimize(level=2).n_2q_gates()`, not the number off the block you built.

## 7. Stage-explicit: what actually executes

Optimization is only half of it. On real connectivity, distant two-qubit gates need `SWAP`s, and
those SWAPs are then decomposed into the target gate set. `qarp.resources.estimate()` runs the
*real* pipeline and snapshots a counted vector at each stage boundary — these are ground truth,
not a parallel cost model.

In [ ]:
def fanout(n=5):
    b = SimpleBlock(n, name="fanout")
    b.h(0)
    for k in range(1, n):
        b.cx(0, k)                    # a star, on a line architecture -> SWAPs
    b.rz(3, 0.3)
    return b.build()


blk = fanout()
device = Device(5, architecture=get_nearest_neighbour_architecture(1, 5))
report = estimate(blk, gateset=qx.clifford_t_rz_gateset(), device=device, device_label="line-5")

head = f"{'stage':<12} {'qubits':>7} {'depth':>6} {'gates':>6} {'2q':>4} {'swap':>5}"
print(head + "\n" + "-" * len(head))
for stage, v in report.items():
    swap = "—" if v.swap_count is None else v.swap_count
    print(f"{stage.value:<12} {v.n_qubits:>7} {v.depth:>6} {v.n_gates:>6} {v.n_2q:>4} {swap:>5}")

Read the table bottom-up and the story is: 4 two-qubit gates as written, 6 after routing (2 of
them SWAPs), 10 at the target stage because each router SWAP became 3 CX. `swap_count` lives
**only** at `ROUTED` — at `TARGET` there are no SWAP commands left to count, so it is `None`,
and `None` means "not applicable / not measured", never `0`.

The `LOGICAL` stage is exactly the block counters, by construction:

In [ ]:
logical = report[Stage.LOGICAL]
print(f"blk.n_gates()     = {blk.n_gates():>2}   logical.n_gates = {logical.n_gates}")
print(f"blk.n_2q_gates()  = {blk.n_2q_gates():>2}   logical.n_2q    = {logical.n_2q}")
print(f"blk.depth()       = {blk.depth():>2}   logical.depth   = {logical.depth}")

## 8. The whole histogram in one call

`n_gates_of_type` one type at a time gets tedious. Two ways to get everything at once — both
unfiltered, both agreeing with `n_gates_of_type`:

In [ ]:
print("count_ops():", dag.count_ops())

# Straight off the commands. The field is `.gate`, not `.gate_type`.
print("Counter:    ", dict(Counter(qx.gate_name(c.gate) for c in block.flatten())))

# ...and it agrees with the per-type counter, type by type:
print("agrees:     ", all(block.n_gates_of_type(getattr(qx.GateType, name)) == n
                          for name, n in dag.count_ops().items()))

In [ ]:
print("resource-vector histogram (LOGICAL):", logical.op_histogram)

## 9. Cheat sheet

| You want | Use | Filtered? | Needs `build()` |
|---|---|---|---|
| register width | `block.n_qubits` *(attribute)* | — | no |
| classical width | `block.n_cbits` *(attribute)* | — | **yes** — 0 until built |
| free parameters | `block.symbols` *(sorted tuple)* | — | yes |
| total gates | `block.n_gates()` | physical only | yes |
| 1q / 2q / k-qubit | `n_1q_gates()`, `n_2q_gates()`, `n_nqb_gates(k)` | physical only | yes |
| one specific gate | `block.n_gates_of_type(qx.GateType.CX)` | **no** | yes |
| every gate at once | `qx.CircuitDAG.from_commands(block.flatten()).count_ops()` | **no** | yes |
| raw command count | `len(block.flatten())` | **no** | yes |
| critical path | `block.depth()` | `Barrier`/`GPhase` weigh 0 | yes |
| post-compilation cost | `block.optimize(level=2).n_2q_gates()` | physical only | yes |
| hardware cost incl. SWAPs | `estimate(block, gateset=..., device=...)` | per stage | yes |

Three things to carry away:

- **`n_gates_of_type` is the odd one out** — unfiltered, so it sums to `len(flatten())`, not to
  `n_gates()`. The difference is exactly the non-physical commands.
- **`depth()` is a path, not a count**, and it weighs commands differently again.
- **Counts are of the circuit you wrote.** Anything you quote as a hardware number should come
  from `optimize(level=2)` or, if connectivity matters, from `qarp.resources.estimate()`.
- **One attribute is still premature.** `n_cbits` reads `0` until build on every block, and it
  does not raise — build first, then read it.